# Project 05 — Binary Outcomes (Logistic Regression)

**Scenario.** A ligand either binds (1) or does not bind (0) to a receptor in each of $N$ independent wells. The probability of binding rises with a continuous covariate $x$ (a standardized concentration / hydrophobicity score). We want the posterior over the dose–response relationship.

**New skill.** *Link functions.* We never put a prior on a probability directly. We model a linear predictor $\eta=\alpha+\beta x$ on the unconstrained log-odds scale and squash it through the logistic sigmoid. The crucial consequence: priors live on the log-odds scale, and we must read their implications back **on the probability scale**.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Each well is an independent Bernoulli trial whose success probability is a logistic function of $x$:

$$\operatorname{logit}(p_i)=\alpha+\beta x_i,\qquad y_i\sim\text{Bernoulli}(p_i).$$

**Assumptions made explicit:** (a) wells are independent; (b) the log-odds is *linear* in $x$; (c) $x$ is measured without error; (d) no omitted covariate confounds the relationship. We standardize $x$ so $\alpha$ is the log-odds of binding at the mean covariate, which makes priors interpretable. Truth: $\alpha=0.3$ ($p\approx0.57$ at mean $x$), $\beta=1.4$ (each $+1$ SD multiplies the odds by $e^{1.4}\approx4.1$).

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']
print(f"n={data['n']} wells, binding rate={y.mean():.3f}, "
      f"true alpha={data['truth']['alpha']}, beta={data['truth']['beta']}")

## Step 2 — Model specification (likelihood, link, justified priors)

$$y_i\sim\text{Bernoulli}(p_i),\quad p_i=\operatorname{logit}^{-1}(\alpha+\beta x_i),\quad \alpha,\beta\sim\text{Normal}(0,1.5).$$

**Why Normal(0, 1.5) and not the 'non-informative' Normal(0, 10)?** This is the heart of the project. A prior on the *log-odds* coefficient maps to a prior on the *probability*. A wide Normal(0, 10) on $\alpha$ says the binding probability at mean $x$ is almost certainly either $\approx 0$ or $\approx 1$ — a bimodal U-shape piled at the edges — which is an absurd belief to hold *before* seeing data. Normal(0, 1.5) spreads the implied probability sensibly across $(0,1)$. We verify this next.

In [ ]:
from model import build_model, fit
model = build_model(data, prior_sd=1.5)
model

## Step 3 — Prior predictive check ON THE PROBABILITY SCALE

We draw coefficients from each candidate prior, push them through the link, and look at the implied binding probability $p$ at the mean covariate ($x=0$, so $p=\operatorname{logit}^{-1}(\alpha)$). A good prior spreads $p$ across the unit interval; a bad one piles it at the 0/1 edges.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))
rng = np.random.default_rng(RNG)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, sd, ttl in [(axes[0], 1.5, 'Normal(0, 1.5) — sensible'),
                    (axes[1], 10.0, 'Normal(0, 10) — pathological')]:
    a = rng.normal(0, sd, size=20000)
    p0 = sigmoid(a)
    ax.hist(p0, bins=40, color='#4C72B0', edgecolor='white', density=True)
    ax.set(xlabel='implied p at mean x', title=ttl)
axes[0].set_ylabel('density')
plt.tight_layout()

Read it: **left** (Normal(0, 1.5)) spreads the implied probability broadly and unimodally around 0.5 — exactly the agnostic belief we want. **Right** (Normal(0, 10)) is a U pinned at 0 and 1: the 'vague' prior secretly asserts the assay is near-deterministic. Wide priors are *not* uninformative once a nonlinear link is involved.

In [ ]:
# Full prior predictive across all x (not just the mean), default prior.
with model:
    prior = pm.sample_prior_predictive(draws=400, random_seed=RNG)
pp_p = sigmoid(prior.prior['alpha'].values[..., None]
               + prior.prior['beta'].values[..., None] * x[None, None, :])
rate = pp_p.mean(axis=-1).ravel()
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(rate, bins=30, color='#55A868', edgecolor='white')
ax.set(xlabel='prior-implied overall binding rate', ylabel='count',
       title='Normal(0,1.5) prior predictive — broad, no edge pile-up')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=1000, tune=1000, chains=4`. Logistic GLMs with standardized predictors have benign geometry, so default NUTS is plenty. Four chains give reliable split-$\hat R$; we fix `random_seed`.

In [ ]:
idata = fit(data, prior_sd=1.5, draws=1000, tune=1000, chains=4, seed=101)

## Step 5 — Computational diagnostics

Check **$\hat R$** ($\approx1.00$), **ESS** (bulk/tail $\gtrsim 400$) and **divergences** (expect 0). The trace should be well-mixed fuzzy caterpillars. If $\hat R$ stayed high we would suspect $\alpha$–$\beta$ correlation from un-centered $x$ (cured by standardizing, which we did).

In [ ]:
print(az.summary(idata, var_names=['alpha', 'beta']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_trace(idata, var_names=['alpha', 'beta']); plt.tight_layout()

## Step 6 — Posterior predictive checks

Two views. First the ArviZ PPC of the 0/1 outcomes; then the **calibration** view that matters for a GLM — bin wells by predicted probability and check the observed binding fraction tracks the diagonal.

In [ ]:
az.plot_ppc(idata, num_pp_samples=200); plt.tight_layout()

In [ ]:
p_hat = idata.posterior['p'].mean(('chain', 'draw')).values
bins = np.linspace(0, 1, 7)
idx = np.digitize(p_hat, bins) - 1
xs, ys = [], []
for b in range(len(bins) - 1):
    m = idx == b
    if m.sum() >= 3:
        xs.append(p_hat[m].mean()); ys.append(y[m].mean())
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='perfect calibration')
ax.plot(xs, ys, 'o-', color='#C44E52', label='binned observed')
ax.set(xlabel='predicted p', ylabel='observed binding fraction',
       title='Calibration of the logistic fit')
ax.legend(); plt.tight_layout()

## Step 7 — Model criticism & prior sensitivity

We compare the recovered coefficients to the known truth, and re-fit under a tight, weak, and vague prior to confirm the posterior is robust *given data* — even though the vague prior was pathological a priori.

In [ ]:
from shared.bayes_utils import check_recovery
for res in check_recovery(idata, data['truth']):
    print(res)

In [ ]:
rows = []
for sd in (0.5, 1.5, 10.0):
    ida = fit(data, prior_sd=sd, draws=500, tune=500, chains=2, seed=11)
    s = az.summary(ida, var_names=['beta'], hdi_prob=0.94).loc['beta']
    rows.append((sd, s['mean'], s['sd']))
    print(f'prior_sd={sd:>4}: beta mean={s["mean"]:.3f} sd={s["sd"]:.3f}')

## Step 8 — Decision & communication

Translate the posterior into the dose–response a collaborator needs: the probability that $\beta>0$ (binding genuinely increases with $x$) and the covariate value at which binding crosses 50%.

In [ ]:
beta_post = idata.posterior['beta'].values.ravel()
alpha_post = idata.posterior['alpha'].values.ravel()
p_pos = float(np.mean(beta_post > 0))
x50 = -alpha_post / beta_post
lo, hi = np.percentile(x50, [3, 97])
print(f'P(beta > 0 | data) = {p_pos:.3f}')
print(f'x at p=0.5: median={np.median(x50):.2f} SD, 94% [{lo:.2f}, {hi:.2f}]')

**Conclusion (for a collaborator).** Binding probability increases strongly and almost-certainly with the covariate ($P(\beta>0)\approx1$). The half-maximal point sits near the mean covariate value. See `summary_onepager.md` for the decision framing.